# **1. Perkenalan Dataset**


Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:

1. **Sumber Dataset**:  
   Dataset diperoleh dari Kaggle: *Credit Card Fraud Detection* (https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud).  
   Dataset ini berisi 284.807 transaksi kartu kredit nasabah Eropa pada September 2013.  
   Kolom `V1`–`V28` adalah hasil PCA (anonim), `Time` dan `Amount` adalah data asli, dan `Class` adalah label (0 = normal, 1 = fraud).

2. **Jumlah Data**:  
   - 284.807 baris, 31 kolom  
   - Hanya 492 transaksi fraud (≈ 0.172%) — **sangat tidak seimbang (imbalanced)**.

3. **Masalah Utama**:  
   - Class imbalance ekstrem (fraud hanya 0.17% data).  
   - Fitur PCA tidak dapat diinterpretasi secara langsung.  
   - Skala `Time` dan `Amount` sangat berbeda dari fitur PCA.


# **2. Import Library**


Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

print('Library imported successfully.')

# **3. Memuat Dataset**

Dataset dimuat dari file CSV. Cek struktur awal untuk memahami bentuk data.


In [ ]:
df = pd.read_csv('../creditcard_raw/creditcard.csv')

print('Shape:', df.shape)
print('\n5 sample baris:')
display(df.head())
print('\nTipe data:')
display(df.dtypes)

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

## 4.1 Distribusi Target (Class)

In [ ]:
class_counts = df['Class'].value_counts()
print('Distribusi Class:')
print(class_counts)
print(f'\nPersentase fraud: {class_counts[1]/len(df)*100:.4f}%')

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(data=df, x='Class', ax=ax[0])
ax[0].set_title('Count Plot')
ax[0].set_xticklabels(['Normal (0)', 'Fraud (1)'])

colors = ['#4CAF50', '#F44336']
ax[1].pie(class_counts.values, labels=['Normal', 'Fraud'],
          autopct='%1.2f%%', colors=colors, startangle=90)
ax[1].set_title('Distribution')
plt.tight_layout()
plt.show()

**Temuan:** Dataset sangat tidak seimbang — fraud hanya 0.17%. Ini akan menjadi tantangan utama dalam preprocessing dan modeling nantinya.

## 4.2 Distribusi Time dan Amount

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0,0].hist(df['Time'], bins=50, edgecolor='black', alpha=0.7)
axes[0,0].set_title('Distribusi Time (detik)')
axes[0,0].set_xlabel('Time (detik sejak transaksi pertama)')

axes[0,1].hist(df['Amount'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0,1].set_title('Distribusi Amount')
axes[0,1].set_xlabel('Amount')

axes[1,0].hist(df[df['Class']==0]['Amount'], bins=50, alpha=0.7, label='Normal', color='green')
axes[1,0].hist(df[df['Class']==1]['Amount'], bins=50, alpha=0.7, label='Fraud', color='red')
axes[1,0].set_title('Amount by Class')
axes[1,0].legend()

axes[1,1].boxplot([df[df['Class']==0]['Amount'], df[df['Class']==1]['Amount']],
                  labels=['Normal', 'Fraud'])
axes[1,1].set_title('Boxplot Amount by Class')
axes[1,1].set_ylabel('Amount')

plt.tight_layout()
plt.show()

print('Statistik Amount per Class:')
print(df.groupby('Class')['Amount'].describe())

**Temuan:**
- `Time` terdistribusi cukup merata sepanjang 2 hari (172.792 detik ≈ 48 jam).
- `Amount` sangat right-skewed — mayoritas transaksi kecil (median $22), dengan beberapa outlier besar hingga $25.691.
- Transaksi fraud cenderung memiliki amount yang lebih kecil dibanding normal.

## 4.3 Missing Values dan Duplikasi

In [ ]:
print('Missing values per kolom:')
print(df.isnull().sum().to_string())
print(f'\nTotal missing: {df.isnull().sum().sum()}')
print(f'\nJumlah duplikat: {df.duplicated().sum()}')

**Temuan:** Tidak ada missing values. Ada 1.081 baris duplikat yang perlu di-handle nanti.

## 4.4 Korelasi Fitur PCA terhadap Class

In [ ]:
corr_with_class = df.corr()['Class'].drop('Class').sort_values()

plt.figure(figsize=(10, 8))
colors = ['red' if v < 0 else 'green' for v in corr_with_class.values]
corr_with_class.plot(kind='barh', color=colors)
plt.title('Korelasi Fitur terhadap Class')
plt.xlabel('Koefisien Korelasi')
plt.axvline(0, color='black', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

print('Korelasi tertinggi (positif):')
print(corr_with_class.sort_values(ascending=False).head(5))
print('\nKorelasi terendah (negatif):')
print(corr_with_class.sort_values(ascending=True).head(5))

**Temuan:**
- V11, V4, V2, V21 memiliki korelasi positif tertinggi dengan fraud.
- V17, V14, V12, V10 memiliki korelasi negatif tertinggi.
- `Time` dan `Amount` hampir tidak berkorelasi dengan `Class`.
- **Korelasi sangat rendah** karena sifat data yang sangat imbalanced — ini wajar dan bukan masalah.

## 4.5 Deteksi Outlier pada Amount

In [ ]:
Q1 = df['Amount'].quantile(0.25)
Q3 = df['Amount'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['Amount'] < lower) | (df['Amount'] > upper)]
print(f'Batas outlier (IQR): lower={lower:.2f}, upper={upper:.2f}')
print(f'Jumlah outlier Amount: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)')
print(f'\nStatistik outlier:')
print(outliers['Amount'].describe())

**Temuan:** Terdapat banyak outlier pada `Amount`. Namun, karena fraud sering terjadi pada transaksi dengan nilai yang tidak biasa, penghapusan outlier BUKAN langkah yang bijak. Kita akan menggunakan **RobustScaler** untuk `Amount` agar outlier tidak mendominasi scaling.

## 4.6 Statistik Deskriptif V1-V28

In [ ]:
v_cols = [f'V{i}' for i in range(1, 29)]
print('Statistik deskriptif V1-V28:')
display(df[v_cols].describe().round(4))

**Temuan:** Semua fitur V1-V28 memiliki mean ≈ 0 dan std ≈ 1. Ini menandakan data sudah ternormalisasi (hasil PCA). Tidak perlu scaling ulang untuk fitur-fitur ini.

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

**Langkah yang dilakukan:**
1. Menangani duplikasi.
2. Scaling `Time` (StandardScaler) dan `Amount` (RobustScaler karena outlier).
3. Memisahkan fitur dan target.
4. Stratified train-test split (menjaga proporsi fraud).
5. SMOTE oversampling pada data training.
6. Menyimpan hasil preprocessing.


## 5.1 Menangani Duplikasi

In [ ]:
print(f'Duplikat sebelum: {df.duplicated().sum()}')
df_clean = df.drop_duplicates()
print(f'Duplikat setelah: {df_clean.duplicated().sum()}')
print(f'Shape setelah drop duplicates: {df_clean.shape}')

**Reasoning:** Duplikat pada data transaksi kartu kredit bisa terjadi karena entry ganda atau pengambilan data yang redundant. Kita hapus untuk menghindari data leakage.

## 5.2 Scaling Time dan Amount

In [ ]:
from sklearn.preprocessing import RobustScaler

X = df_clean.copy()

scaler_time = StandardScaler()
X['Time'] = scaler_time.fit_transform(X[['Time']])

scaler_amount = RobustScaler()
X['Amount'] = scaler_amount.fit_transform(X[['Amount']])

print('Time setelah StandardScaler:')
print(f'  Mean: {X["Time"].mean():.4f}, Std: {X["Time"].std():.4f}')
print('Amount setelah RobustScaler:')
print(f'  Median: {X["Amount"].median():.4f}, IQR-range approx: {X["Amount"].quantile(0.75)-X["Amount"].quantile(0.25):.4f}')

**Reasoning:**
- `Time` tidak memiliki outlier ekstrem, cukup pakai StandardScaler.
- `Amount` sangat right-skewed dengan banyak outlier. RobustScaler (berbasis median dan IQR) lebih cocok daripada StandardScaler karena tidak terpengaruh outlier.
- V1-V28 sudah ternormalisasi dari PCA (mean≈0, std≈1), tidak perlu di-scale ulang.

## 5.3 Memisahkan Fitur dan Target

In [ ]:
y = X['Class']
X_features = X.drop('Class', axis=1)

print('X shape:', X_features.shape)
print('y shape:', y.shape)
print(f'Proporsi fraud di y: {y.mean()*100:.4f}%')

## 5.4 Stratified Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')
print(f'\nDistribusi train:\n{y_train.value_counts().to_string()}')
print(f'Fraud % di train: {y_train.mean()*100:.4f}%')
print(f'\nDistribusi test:\n{y_test.value_counts().to_string()}')
print(f'Fraud % di test: {y_test.mean()*100:.4f}%')

**Reasoning:** Stratify wajib digunakan karena data sangat imbalanced (fraud hanya 0.17%). Tanpa stratify, ada risiko test set tidak mengandung fraud sama sekali, yang akan membuat evaluasi model tidak bermakna.

## 5.5 SMOTE Oversampling (Training Set Only)

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print('Setelah SMOTE:')
train_res_counts = y_train_res.value_counts()
print(train_res_counts)
print(f'Rasio: {train_res_counts[0] / train_res_counts[1]:.2f}:1')

**Reasoning:** SMOTE membuat synthetic samples untuk kelas minoritas (fraud) di training set, sehingga model tidak bias ke kelas mayoritas. SMOTE hanya diterapkan pada training set untuk menghindari data leakage ke test set.

## 5.6 Menyimpan Hasil Preprocessing

In [ ]:
import os

output_dir = 'creditcard_preprocessing'
os.makedirs(output_dir, exist_ok=True)

pd.DataFrame(X_train, columns=X_features.columns).to_csv(
    f'{output_dir}/X_train.csv', index=False)
pd.DataFrame(X_test, columns=X_features.columns).to_csv(
    f'{output_dir}/X_test.csv', index=False)
pd.DataFrame(X_train_res, columns=X_features.columns).to_csv(
    f'{output_dir}/X_train_resampled.csv', index=False)
pd.DataFrame(y_train, columns=['Class']).to_csv(
    f'{output_dir}/y_train.csv', index=False)
pd.DataFrame(y_test, columns=['Class']).to_csv(
    f'{output_dir}/y_test.csv', index=False)
pd.DataFrame(y_train_res, columns=['Class']).to_csv(
    f'{output_dir}/y_train_resampled.csv', index=False)

print('Semua file preprocessing tersimpan di folder:', output_dir)
print(f'\nDaftar file:')
for f in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, f)
    fsize = os.path.getsize(fpath)
    print(f'  {f}: {fsize/1024:.1f} KB')

## 5.7 Ringkasan Preprocessing

**Kesimpulan Preprocessing:**
- **Duplikasi**: 1.081 baris duplikat dihapus.
- **Missing Values**: Tidak ada.
- **Scaling**: `Time` → StandardScaler, `Amount` → RobustScaler (karena outlier). V1-V28 tidak di-scale (sudah normal dari PCA).
- **Train/Test Split**: 80/20, stratified by Class.
- **SMOTE**: Synthetic oversampling pada training set untuk menyeimbangkan kelas.
- **Output**: 6 file CSV siap pakai untuk modeling.

Dataset preprocessing ini siap digunakan di repositori **Membangun_Model** (Kriteria 2) dan **Workflow-CI** (Kriteria 3).